# ♟️ Chess Expert vs Stockfish — win reel (Colab GPU)

Plays the model against Stockfish across several Elo caps, saves every **win**, and stitches the wins into one **cool highlight-reel MP4** for LinkedIn.

1. `Runtime → Change runtime type →` **GPU** (L4/A100 makes the search much faster).
2. Run the cells top-to-bottom, then download the MP4.

> ⚠️ **Depth reality:** the search calls the network at every node and branches wide, so cost grows **exponentially**. **depth 4–6** is the realistic range (depth 6 already beats ~1500). depth ~12 will not finish. Start with `--depth 4` and fewer games to gauge the time.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Setup — code, deps, and Stockfish

In [ ]:
REPO_URL = "https://github.com/AhPro7/chess-expert.git"  # <-- your repo
import os
if not os.path.isdir("chess-expert"):
    !git clone $REPO_URL
%cd chess-expert
!git pull
!pip -q install -r requirements.txt
# apt installs stockfish to /usr/games, which is NOT on Colab's PATH.
# Symlink it so the bare `stockfish` command resolves everywhere.
!apt-get -qq install -y stockfish >/dev/null
!ln -sf "$(ls /usr/games/stockfish /usr/bin/stockfish 2>/dev/null | head -1)" /usr/local/bin/stockfish
!which stockfish && echo "stockfish ready" || echo "STOCKFISH NOT FOUND"


## 2. Run the tournament (saves every win)
Model vs Stockfish at each Elo, `--games` each, alternating colours. Wins are saved to `demo/wins/`.

Tip: to gauge timing, first try `--elos 1500 --games 1 --depth 4`. Then scale up.

**Going deeper (depth 6–8)?** Search cost is exponential — *narrow the tree* to keep it usable:
add **`--branch 3 --max-cand 8`** (fast-ish) or **`--branch 2 --max-cand 6`** (fastest, a bit weaker).
Without them, depth 8 can take many minutes per move.


In [ ]:
!python scripts/tournament.py \
  --elos 1300 1400 1500 1600 1700 \
  --games 3 \
  --depth 5 \
  --device cuda \
  --out demo/wins

## 3. Build the highlight reel (MP4) and download it

In [ ]:
!python scripts/win_reel.py --wins demo/wins --out demo/wins_reel --fps 3
from google.colab import files
files.download('demo/wins_reel.mp4')

## (optional) Also measure an Elo estimate
How the model scores vs each capped Elo — a number for the post.

In [ ]:
!python scripts/vs_stockfish.py --elos 1500 1700 --games 6 --depth 4